**Assumptions**

- Data may not be standardised between different file versions
- Data may have null or non-conforming values
- Raw data must be preserved at Bronze layer, standardised at silver layer, business consumable at gold layer

**Part 1**

**Customers**

In [0]:
#Explore data to understand form and grain
df_customers_1 = spark.read.json("/Volumes/tech_assessment/bean_counter/landing/customers/customers_batch1.json")
df_customers_1.display()

In [0]:
df_customers_2 = spark.read.json("/Volumes/tech_assessment/bean_counter/landing/customers/customers_batch2.json")
df_customers_2.display()

In [0]:
from pyspark.sql.functions import col, current_timestamp

#Build a databricks volume path
catalog = "tech_assessment"
schema = "bean_counter"
volume = "landing"

base = f"/Volumes/{catalog}/{schema}/{volume}"

#Set source data to auto load
source_path = f"{base}/customers"

#Set schema path
schema_path = f"{base}/_schemas/bronze_customers"


checkpoint_path = f"{base}/_checkpoints/bronze_customers"

#Set target bronze delta table
target_table = f"{catalog}.{schema}.bronze_customers"

#Configure stream / autoload
(
    spark.readStream
        .format("cloudFiles")
        #Specify file type
        .option("cloudFiles.format", "json")
        #Specify schema location
        .option("cloudFiles.schemaLocation", schema_path)
        #Set schema evolution mode as per requirement - Make sure the new marketing_opt_in column in customers_batch2.json does not break the pipeline.
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_path)
        #Add source path & ingestion timestamp as per requirement
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingested_at", current_timestamp())
        .writeStream
        #Set checkpoint path to ensure only new files are processed on rerun as per requirement
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
)

In [0]:
df_bronze_customers = spark.table("tech_assessment.bean_counter.bronze_customers")

df_bronze_customers.display()

**Orders**

In [0]:
#Explore source data
df_orders_1 = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv("/Volumes/tech_assessment/bean_counter/landing/orders/orders_2025-03-01.csv")
)

display(df_orders_1)

In [0]:
df_orders_2 = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv("/Volumes/tech_assessment/bean_counter/landing/orders/orders_2025-03-02.csv")
)

display(df_orders_2)

In [0]:
df_orders_3 = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv("/Volumes/tech_assessment/bean_counter/landing/orders/orders_2025-03-03.csv")
)

display(df_orders_3)

In [0]:
from pyspark.sql.functions import col, current_timestamp

#Build a databricks volume path
catalog = "tech_assessment"
schema = "bean_counter"
volume = "landing"

base = f"/Volumes/{catalog}/{schema}/{volume}"

source_path = f"{base}/orders"
schema_path = f"{base}/_schemas/bronze_orders"
checkpoint_path = f"{base}/_checkpoints/bronze_orders"
target_table = f"{catalog}.{schema}.bronze_orders"

(
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(source_path)
        #Add source path & ingestion timestamp as per requirement
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingested_at", current_timestamp())
        .writeStream
        #Set checkpoint path to ensure only new files are processed on rerun as per requirement
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
)

In [0]:
df_bronze_orders = spark.table("tech_assessment.bean_counter.bronze_orders")

df_bronze_orders.display()

**Part 2**

**Orders**
1. Cast to proper types and parse both timestamp formats (yyyy-MM-dd HH:mm:ss and
dd/MM/yyyy HH:mm) into one timestamp column.
2. Remove exact duplicate rows.
3. Fill a missing unit_price from products.csv.

4. Send rows with quantity &lt;= 0 or a missing or empty customer_id to a quarantine table with a
rejection reason. Do not delete them.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    BooleanType,
    TimestampType,
)

In [0]:

df_bronze_orders.display()

Cast to proper types and parse both timestamp formats (yyyy-MM-dd HH:mm:ss and
dd/MM/yyyy HH:mm) into one timestamp column.

In [0]:
from pyspark.sql.functions import col, coalesce, to_timestamp, try_to_timestamp, lit

silver_orders = (
    df_bronze_orders
        .select(
            "order_id",
            "customer_id",
            "store_id",
            "product_id",
            #Cast quantity as integer
            col("quantity").cast("integer"),
            #Cast unit_price as decimal as this is financial data
            col("unit_price").cast("decimal(10,2)"),
            "order_ts",
            "_rescued_data",
            "_source_file",
            "_ingested_at"        
            )
        #Perform a coalesce to convert order_ts to timestamp based on both formats being present
        .withColumn(
                "order_ts",
                coalesce(
                    try_to_timestamp(col("order_ts"), lit("yyyy-MM-dd HH:mm:ss")),
                    try_to_timestamp(col("order_ts"), lit("dd/MM/yyyy HH:mm")),
                )
        )
)
#Display for debug
silver_orders.display()

In [0]:
#Built in function to drop duplicates
df_silver_orders_deduplicated = silver_orders.dropDuplicates()

df_silver_orders_deduplicated.display()

In [0]:
#Explore products data and load to data frame
df_products = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv("/Volumes/tech_assessment/bean_counter/landing/products.csv")
)

from pyspark.sql.functions import col, coalesce

#make a new data frame with product_id and renamed column to prevent ambiguity
product_prices = df_products.select(
    "product_id",
    col("unit_price").alias("reference_unit_price"),
)

#Join silver orders with product_prices to get reference unit price and coalesce to get unit price if not present
silver_orders_unit_price = (
    df_silver_orders_deduplicated
        .join(product_prices, "product_id", "left")
        .withColumn(
            "unit_price",
            coalesce("unit_price", "reference_unit_price"),
        )
        .drop("reference_unit_price")
)
#Build a rollup data frame with correct column ordinal positions
silver_orders_unit_price = silver_orders_unit_price.select(
    "order_id",
    "customer_id",
    "store_id",
    "product_id",
    "quantity",
    "unit_price",
    "order_ts",
    "_rescued_data",
    "_source_file",
    "_ingested_at",
)

silver_orders_unit_price.display()

**Send rows with quantity <= 0 or a missing or empty customer_id to a quarantine table with a rejection reason. Do not delete them.**

In [0]:
filtered_df = (
    silver_orders_unit_price
        #Filter Data frame to identify quarantined rows
        .filter(
            (col("quantity") <= 0)
            | col("customer_id").isNull()
            | (col("customer_id") == "None")
            | (col("customer_id") == "")
        )
        #Build up a reject reason column
        .withColumn(
            "rejection_reason",
            when(
                col("quantity") <= 0,
                lit("Invalid Quantity: must be greater than zero"),
            ).when(
                col("customer_id").isNull()
                | (col("customer_id") == "None")
                | (col("customer_id") == ""),
                lit("Missing customer ID"),


            )
        )
)

display(filtered_df)
#Save as a table for review
filtered_df.write.format("delta").mode("append").saveAsTable("tech_assessment.bean_counter.silver_orders_quarantine")

In [0]:
#Write our valid orders to a silver table based on above validation requirements
valid_orders = silver_orders_unit_price.filter(
    (col("quantity") > 0)
    & col("customer_id").isNotNull()
    & (col("customer_id") != "None")
    & (col("customer_id") != "")
)

(
    valid_orders.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("tech_assessment.bean_counter.silver_orders")
)

**Customers**

Keep one current row per customer_id (latest updated_at) using MERGE. Running the pipeline
twice must not create duplicates.

In [0]:
df_customers = spark.table("tech_assessment.bean_counter.bronze_customers")
df_customers.display()

In [0]:
#Initially we will perform a full load of data
#df_customers.write.format("delta").mode("append").saveAsTable("tech_assessment.bean_counter.silver_customers")

In [0]:
#Perform a merge to update the silver table with the bronze table, this will allow us to incrementally update the silver table
df_customers.createOrReplaceTempView("source_customers")

spark.sql("""
    MERGE INTO tech_assessment.bean_counter.silver_customers AS target
    USING source_customers AS source
    ON target.customer_id = source.customer_id

    WHEN MATCHED
         AND source.updated_at > target.updated_at
    THEN UPDATE SET *

    WHEN NOT MATCHED
    THEN INSERT *
""")

**Part 3: Gold Layer in SQL**

In [0]:

%sql
select * from tech_assessment.bean_counter.silver_orders

**gold_daily_sales: revenue, units and distinct orders per day per store.**

In [0]:
%sql
CREATE OR REPLACE TABLE tech_assessment.bean_counter.gold_daily_sales AS

SELECT
    CAST(order_ts AS DATE) AS order_date,
    store_id,
    --Calculate revenue
    SUM(unit_price * quantity) AS revenue,
    --Calculate total units sold
    SUM(quantity) AS total_units,
    --Identify only distinct orders
    COUNT(DISTINCT order_id) AS distinct_orders
FROM tech_assessment.bean_counter.silver_orders
GROUP BY
    CAST(order_ts AS DATE),
    store_id

In [0]:
%sql
select * from tech_assessment.bean_counter.gold_daily_sales

**gold_product_ranking: revenue by product, with each product&#39;s rank within its category (use
a window function).**

In [0]:
%sql
CREATE OR REPLACE TABLE tech_assessment.bean_counter.gold_product_ranking AS
--Create our temporary result set
WITH product_revenue AS (

    SELECT
        products.product_id,
        products.product_name,
        products.category,
        SUM(orders.unit_price * orders.quantity) AS revenue
    FROM tech_assessment.bean_counter.silver_products products
    INNER JOIN tech_assessment.bean_counter.silver_orders orders
        ON products.product_id = orders.product_id
    GROUP BY
        products.product_id,
        products.product_name,
        products.category

)
--use SQL rank function to rank products by revenue
SELECT
    product_id,
    product_name,
    category,
    revenue,
    RANK() OVER (
        PARTITION BY category
        ORDER BY revenue DESC
    ) AS product_rank
FROM product_revenue
